# S2SVV5 Processing

- Whole genome sequence data; Illumina and Nanopore.
- Shipped and submitted to Plasmidsaurus 01/05/2026.
- Data available on: 01/14/2026.
- Started procecessing on: 01/15/2026.

## Steps
1. Download, data organization, file renaming
2. Creating seqsamples and cross-checking in LIMS.
3. QA/QC
4. Breseq Pipeline and Breseq analysis

## 0. Set-up

In [ ]:
## Make sure running in aisynbio_env

In [1]:
# Reload magic command to ensure that changes made to my imported modules are being picked up by the notebook continuously

%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

# Add project root to path for access to workflows and tasks
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [3]:
import os

# Must add environment's bin to the PATH inside the notebook
env_bin = os.path.join(os.path.dirname(sys.executable), "")
os.environ["PATH"] = env_bin + ":" + os.environ["PATH"]

## 1. Download, file organization, renaming
- Create seqorder name and makedir into reception folder (/synbio/ai_synbio_data/experimental_data/downloads - should this be temporary?). Also make homedir to copy and view analysis reports.
- Download data into folder.
- Spot-check if duplicate read ID problem is fixed.
- Create new seqorder folder in experimental_data/sequencing_data/ and respective libraries.
- Copy all nanopore fastqs into long lib, renaming in the process.
- Copy all illumina fastqs into short lib, renaming in the process.
Creating sample seqs and cross-checking with LIMS


In [4]:
# Create seqorder name, make reception folder and seqorder analysis folder in nspahr homedir

from aisynbiopipeline.workflows.plasmidsaurus import create_seqorder_name
from aisynbiopipeline.workflows.seq_folder_utils import list_seqorders, SeqOrder, Library, SeqSample
import os

item_code = 'S2SVV5'
seqorder_name = create_seqorder_name(item_code)

reception_dir = '/storage/synbio/ai_synbio_data/experimental_data/downloads/' + seqorder_name
os.makedirs(reception_dir)
print(reception_dir)

home_dir = '/storage/nspahr/lib_analysis/' + seqorder_name
breseq_ACN2821_dir = home_dir + '/breseq_analysis_ACN2821'
breseq_ACN3500_dir = home_dir + '/breseq_analysis_ACN3500'
breseq_mixtures_dir = home_dir + '/breseq_analysis_mixtures'
os.makedirs(home_dir, exist_ok=True) 
os.makedirs(breseq_ACN2821_dir, exist_ok=True)
os.makedirs(breseq_ACN3500_dir, exist_ok=True)
os.makedirs(breseq_mixtures_dir, exist_ok=True)

/storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-01-13_S2SVV5


In [5]:
# Download data into folder

from aisynbiopipeline.workflows.plasmidsaurus import get_access_token, download_results, get_credentials

CLIENT_ID = get_credentials("PLASMIDSAURUS_CLIENT_ID")
CLIENT_SECRET = get_credentials("PLASMIDSAURUS_CLIENT_SECRET")
access_token = get_access_token(CLIENT_ID, CLIENT_SECRET)
download_results(item_code, access_token, reception_dir)

ITEM S2SVV5
{'code': 'S2SVV5',
 'done_date': '2026-01-13T23:26:49.370360+00:00',
 'gross': 1620.0,
 'order_name': '',
 'product_name': 'hybrid_extraction',
 'quantity': 9,
 'status': 'complete'}



DOWNLOADING RESULTS FOR S2SVV5 


File downloaded successfully: /storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-01-13_S2SVV5/S2SVV5_results.zip
Unzipping /storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-01-13_S2SVV5/S2SVV5_results.zip to /storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-01-13_S2SVV5/S2SVV5_results
DOWNLOADING READS FOR S2SVV5 


File downloaded successfully: /storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-01-13_S2SVV5/S2SVV5_reads.zip
Unzipping /storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-01-13_S2SVV5/S2SVV5_reads.zip to /storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-01-13_S2SVV5/S2SVV5_reads


In [ ]:
### TODO: How to ensure that archive was successfully unzipped?? Pipeline log file into home_dir?

In [6]:
# Spot-check if duplicate read ID problem is fixed

plasmidsaurus_read_folder_name = item_code + '_reads'
os.listdir(os.path.join(reception_dir, plasmidsaurus_read_folder_name))

['S2SVV5_6_ANL.stock.ACN3500.colony3_nanopore.fastq.gz',
 'S2SVV5_6_ANL.stock.ACN3500.colony3_illumina_R1.fastq.gz',
 'S2SVV5_6_ANL.stock.ACN3500.colony3_illumina_R2.fastq.gz',
 'S2SVV5_4_ANL.stock.ACN3500.colony1_nanopore.fastq.gz',
 'S2SVV5_4_ANL.stock.ACN3500.colony1_illumina_R1.fastq.gz',
 'S2SVV5_4_ANL.stock.ACN3500.colony1_illumina_R2.fastq.gz',
 'S2SVV5_1_ANL.stock.ACN2821.colony1_nanopore.fastq.gz',
 'S2SVV5_1_ANL.stock.ACN2821.colony1_illumina_R1.fastq.gz',
 'S2SVV5_1_ANL.stock.ACN2821.colony1_illumina_R2.fastq.gz',
 'S2SVV5_8_ADP1col1_ACN3500col1_75-25_nanopore.fastq.gz',
 'S2SVV5_8_ADP1col1_ACN3500col1_75-25_illumina_R1.fastq.gz',
 'S2SVV5_8_ADP1col1_ACN3500col1_75-25_illumina_R2.fastq.gz',
 'S2SVV5_2_ANL.stock.ACN2821.colony2_nanopore.fastq.gz',
 'S2SVV5_2_ANL.stock.ACN2821.colony2_illumina_R1.fastq.gz',
 'S2SVV5_2_ANL.stock.ACN2821.colony2_illumina_R2.fastq.gz',
 'S2SVV5_5_ANL.stock.ACN3500.colony2_nanopore.fastq.gz',
 'S2SVV5_5_ANL.stock.ACN3500.colony2_illumina_R1.fastq.

In [7]:
!gunzip -c {os.path.join(reception_dir, plasmidsaurus_read_folder_name, 'S2SVV5_6_ANL.stock.ACN3500.colony3_illumina_R1.fastq.gz')} | head

@LH01025:31:23GJV7LT3:1:1101:1092:1128 1:N:0:AGTACTCATG+ACCACGACAT
ANTGGTCACCGAACATGCTTACCGAAGATATTGATATTACCTGGAAATTACAACGTGCTGGCTGGGATATCCGCTTTGAGCCAAATGCTCTGGTCTGGATTTTGATGCCTGAAACCTTCCAAGGTTTATGGAAACAGCGTTTACGCTGGGC
+
I#IIIII9IIIIIIIIIIIIIIIIIIIIIIIIIIIIIII9IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII9IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
@LH01025:31:23GJV7LT3:1:1101:1795:1128 1:N:0:AGTACTCATG+ACCACGACAT
GNCTCAACTTGAGCAAGTAAAACGTTATGGCATTAAACCAGACGAAAACACCTTAAATGATGCTGTTTTAAAAGTTGCTAGCCAATCTGGTATTAAGTCATTGAGTGCTTTCCAGCAAAAACTTGATGCAATCGCGCCTGGTACTTATGCT
+
I#9IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
@LH01025:31:23GJV7LT3:1:1101:2572:1128 1:N:0:AGTACTCATG+ACCACGACAT
GNATTGGTGTGATGATTTTTCGACACAATCGATTGAGAAACAGTCTTGGCTGATTCATGAGCTCGTGCATGTTTGGCAGTATCAGCAAGGCATAAAGTTAATCCGAAAAGGAATTTTTGAAAGAAAATATCAGTACGTTTTGCAGCAGGGA

gzip: stdout: Broken pipe


In [8]:
!gunzip -c {os.path.join(reception_dir, plasmidsaurus_read_folder_name, 'S2SVV5_6_ANL.stock.ACN3500.colony3_illumina_R1.fastq.gz')} | grep "@LH01025:31:23GJV7LT3:1:1101:1092:1128 1:N:0:AGTACTCATG+ACCACGACAT"


@LH01025:31:23GJV7LT3:1:1101:1092:1128 1:N:0:AGTACTCATG+ACCACGACAT


**Comment:**

- In this spot check, only found the read ID once in this file. I assume this means that we are not dealing with the read duplication problem seen initially in order P4CYGL.

In [9]:
from aisynbiopipeline.workflows.fastq_utils import create_manifest, parse_illumina_fastq_filename

folder = os.path.join(reception_dir, plasmidsaurus_read_folder_name)

manifest = create_manifest(folder, platform='plasmidsaurus_hybrid')
manifest

,sample_name,nanopore_fastq,fwd_fastq,rvs_fastq
2,S2SVV5_1_ANL.stock.ACN2821.colony1,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
4,S2SVV5_2_ANL.stock.ACN2821.colony2,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
8,S2SVV5_3_ANL.stock.ACN2821.colony3,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
1,S2SVV5_4_ANL.stock.ACN3500.colony1,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
5,S2SVV5_5_ANL.stock.ACN3500.colony2,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
0,S2SVV5_6_ANL.stock.ACN3500.colony3,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
6,S2SVV5_7_ADP1col1_ACN3500col1_50-50,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
3,S2SVV5_8_ADP1col1_ACN3500col1_75-25,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
7,S2SVV5_9_ADP1col1_ACN3500col1_90-10,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...


In [10]:
# Plasmidsaurus provides a sort of sample manifest for download from the seqorder page. (not available with read download through API).
# Downloaded to my laptop, uploaded to home_dir, now copying to reception dir.

import shutil

shutil.copy2(os.path.join(home_dir, f'{item_code}-summary-report.csv'), reception_dir)

'/storage/synbio/ai_synbio_data/experimental_data/downloads/Plasmidsaurus_2026-01-13_S2SVV5/S2SVV5-summary-report.csv'

In [11]:
os.listdir(reception_dir)

['S2SVV5_results.zip',
 'S2SVV5_results',
 'S2SVV5_reads.zip',
 'S2SVV5_reads',
 'S2SVV5-summary-report.csv']

In [12]:
# Create new seqorder folder in experimental_data/sequencing_data/ and libraries

seqorder = SeqOrder(seqorder_name, create=True)
short = Library(seqorder, 'Illumina', create=True)
long = Library(seqorder, 'Nanopore', create=True)

In [13]:
# Identify Illumina/ Nanopore reads and copy into respective library folder

from pathlib import Path
import shutil

reads_path = Path(os.path.join(reception_dir, f'{item_code}_reads'))
i_pattern = "*illumina*.fastq.gz"
n_pattern = "*nanopore*.fastq.gz"
i_files = list(reads_path.glob(i_pattern))
n_files = list(reads_path.glob(n_pattern))

def rename_plasmidsaurus_read_file(file_name):
    aisynbio_filename = ('_').join(file_name.split('_')[2:])
    return aisynbio_filename

for file in i_files:
    plasmidsaurus_basename = os.path.basename(file)
    aisynbio_basename = rename_plasmidsaurus_read_file(plasmidsaurus_basename)
    shutil.copy2(reads_path/plasmidsaurus_basename, short.path/'received'/aisynbio_basename)

for file in n_files:
    plasmidsaurus_basename = os.path.basename(file)
    aisynbio_basename = rename_plasmidsaurus_read_file(plasmidsaurus_basename)
    shutil.copy2(reads_path/plasmidsaurus_basename, long.path/'received'/aisynbio_basename)

## 2. Cross-checking this Plasmidsaurus order seqsamples in LIMS and creating SeqSamples.

In [14]:
short_manifest = short.create_manifest('received')
short_manifest

,sample_name,R1,R2
6,ADP1col1_ACN3500col1_50-50,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
3,ADP1col1_ACN3500col1_75-25,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
7,ADP1col1_ACN3500col1_90-10,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
2,ANL.stock.ACN2821.colony1,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
4,ANL.stock.ACN2821.colony2,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
8,ANL.stock.ACN2821.colony3,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
1,ANL.stock.ACN3500.colony1,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
5,ANL.stock.ACN3500.colony2,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
0,ANL.stock.ACN3500.colony3,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...


In [15]:
# Import LIMS utilities (use util_simple.py for standalone LIMS API usage)

%run util_simple.py

✓ LIMS API loaded successfully


In [25]:
# Run a manual LIMS mirror db sync

from aisynbiopipeline.limsapi.sync import sync_all_sheets

sync_all_sheets()

2026-01-15 16:28:02,825 - lims_sync - INFO - Starting sync operation
2026-01-15 16:28:02,826 - lims_sync - INFO - Connecting to Google Sheets
2026-01-15 16:28:03,650 - lims_sync - INFO - Connecting to database
2026-01-15 16:28:03,741 - lims_sync - INFO - Found 13 worksheets: Experiments, Strains, Conditions, Samples, Measurements, Genes, Measurement_types, DNA_constructs, Primers, dgoA_alleles_new, dgoA_alleles_old, robotic_mt_samples, Strain_stocks_ANL
2026-01-15 16:28:03,742 - lims_sync - INFO - Syncing worksheet: Experiments
2026-01-15 16:28:04,272 - lims_sync - INFO - Retrieved 9 rows from Experiments
2026-01-15 16:28:04,310 - lims_sync - INFO - Inserted 3, updated 0 rows in Experiments
2026-01-15 16:28:04,322 - lims_sync - INFO - Marked 3 rows as deleted in Experiments
2026-01-15 16:28:09,323 - lims_sync - INFO - Syncing worksheet: Strains
2026-01-15 16:28:10,089 - lims_sync - INFO - Retrieved 1020 rows from Strains
2026-01-15 16:28:10,288 - lims_sync - INFO - Inserted 1, updated 

{'start_time': '2026-01-15T16:28:02.825393',
 'end_time': '2026-01-15T16:31:11.636422',
 'success': True,
 'tables_synced': 11,
 'total_rows_inserted': 244,
 'total_rows_updated': 0,
 'total_rows_deleted': 219,
 'errors': ["Error syncing worksheet Samples: Failed to get worksheet data: the header row in the worksheet contains duplicates: ['']To manually set the header row, use the `expected_headers` parameter of `get_all_records()`",
  "Error syncing worksheet robotic_mt_samples: Failed to get worksheet data: the header row in the worksheet contains duplicates: ['']To manually set the header row, use the `expected_headers` parameter of `get_all_records()`"]}

In [26]:
# Cross-checking seq sample measurement names

thisExpLIMSseqsamples_short = query_lims(
    'Measurements',
    filters={'Experiment': 'strain_stocks', 'Type': 'Short_DNA_reads'}
)['Name'].to_list()

thisExpLIMSseqsamples_long = query_lims(
    'Measurements',
    filters={'Experiment': 'strain_stocks', 'Type': 'Long_DNA_reads'}
)['Name'].to_list()

print(f"Are all short Plasmidsaurus seqsamples from order {item_code} in the LIMS?")
print(all([x in thisExpLIMSseqsamples_short for x in short_manifest['sample_name'].to_list()]))
print(f"Are all short LIMS seqsamples with selected filters in Plasmidsaurus order {item_code}?")
print(all([x in short_manifest['sample_name'].to_list() for x in thisExpLIMSseqsamples_short]))

long_manifest = long.create_manifest('received')
print(f"Are all Plasmidsaurus long seqsamples from order {item_code} in the LIMS?")
print(all([x in thisExpLIMSseqsamples_long for x in long_manifest['sample_name'].to_list()]))
print(f"Are all LIMS long seqsamples with selected filters in Plasmidsaurus order {item_code}?")
print(all([x in long_manifest['sample_name'].to_list() for x in thisExpLIMSseqsamples_long]))

Are all short Plasmidsaurus seqsamples from order S2SVV5 in the LIMS?
True
Are all short LIMS seqsamples with selected filters in Plasmidsaurus order S2SVV5?
False
Are all Plasmidsaurus long seqsamples from order S2SVV5 in the LIMS?
True
Are all LIMS long seqsamples with selected filters in Plasmidsaurus order S2SVV5?
False


In [27]:
# Creating batch (list) of short seqsamples for this seqorder

seqsamples = [SeqSample(short, row['sample_name']) for _, row in short_manifest.iterrows()]

## 3. Short reads: QA/QC

- fastp (Celery): Must start running workers first
- MultiQC

In [28]:
# Create the trimmed subfolder
short.create_subfolder('trimmed')

In [ ]:
## To run fastp celery worksers, activate micromamba, then call worker script:
"""
export PATH="/opt/micromamba/bin/:$PATH"
eval "$(micromamba shell hook --shell bash)"
micromamba activate
micromamba activate aisynbio_env
python -m aisynbiopipeline.tasks.fastp_task 1
"""

In [ ]:
## fastp workers running:

# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.fastp_task 1

In [29]:
# Where should this code go?

from pathlib import Path

def get_fastp_params(library, seqsample):

    fwd_in_path = seqsample.received[0]
    fwd_in_file = os.path.basename(fwd_in_path)
    fwd_out_file = fwd_in_file.replace('.fastq.gz', '_trimmed.fastq.gz')
    fwd_out_path = os.path.join(library.path, 'trimmed', fwd_out_file)
    rvs_in_path = seqsample.received[1]
    rvs_in_file = os.path.basename(rvs_in_path)
    rvs_out_file = rvs_in_file.replace('.fastq.gz', '_trimmed.fastq.gz')
    rvs_out_path = os.path.join(library.path, 'trimmed', rvs_out_file)
    
    # Normalize Path → str - Celery tasks only accept certain input data types.
    def norm(x): return str(x) if isinstance(x, Path) else x
    
    fastp_params = {
        'path_to_fwd': norm(fwd_in_path),
        'path_to_rev': norm(rvs_in_path),
        'path_to_fwd_out': norm(fwd_out_path),
        'path_to_rev_out': norm(rvs_out_path),
        'threads': 16,
        'polyG':5
    }

    return fastp_params

In [30]:
from celery import Celery
import os

# Create Celery client
client = Celery(
    'client',
    broker=os.getenv('CELERY_BROKER_URL', 'redis://bioseed_redis:6379/10'),
    backend=os.getenv('CELERY_RESULT_BACKEND', 'redis://bioseed_redis:6379/10')
)

# Submit tasks:

results = []

for sample in seqsamples:
    result = client.send_task(
        'fastp.run',
        kwargs=get_fastp_params(short, sample),
        queue='fastp'
    )
    results.append(result)

In [43]:
for i in results:
    print(i.status)

SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS


In [42]:
all([(r.status=='SUCCESS') for r in results])

True

In [44]:
from aisynbiopipeline.workflows.read_qc import run_multiqc
import shutil

multiqc_report = run_multiqc(short.path / 'trimmed')
multiqc_report_dir = os.path.dirname(multiqc_report)
multiqc_report_file = os.path.basename(multiqc_report)
dst_multiqc_report_file = os.path.join(home_dir, 'trimmed_' + multiqc_report_file)
shutil.copy(multiqc_report, dst_multiqc_report_file)


/// ]8;id=816816;https://multiqc.info\MultiQC]8;;\ v1.32

     version_check | MultiQC Version v1.33 now available!
       file_search | Search path: /storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-01-13_S2SVV5/Plasmidsaurus_2026-01-13_S2SVV5_Illumina/trimmed


        searching | ████████████████████████████████████████ 100% 36/36                                                   html

             fastp | Found 9 reports
     write_results | Data        : multiqc_data
     write_results | Report      : multiqc_report.html
           multiqc | MultiQC complete


'/storage/nspahr/lib_analysis/Plasmidsaurus_2026-01-13_S2SVV5/trimmed_multiqc_report.html'

## 4. Short reads: Breseq (population mode)

Copy 3500 reference genome to reference genome folder.

Run breseq:
1. ACN2821 samples (seqsamples2821): breseq against ADP1 (to ensure deletion of DAHP synthases), 2821 ref genome (to ensure presence and identity of dgoA allele)
2. ACN3500 samples and mixture samples (seqsamples3500_and_mix): breseq against ADP1 (to ensure deletion of two IS1236 elements and the DNA between them; 3,475 bp), 3500 ref genome (to ensure presence and seq identity of ver cassette)

In [ ]:
# Copy reference genome into ref genome folder.

from aisynbiopipeline.workflows.reference_utils import get_ref_genomes_path, list_reference_genomes

genbank_file_src = '/storage/nspahr/tmp/ACN3500_IRZ.gbk'
genbank_file_basename = os.path.basename(genbank_file_src)
genbank_file_dst = os.path.join(get_ref_genomes_path(), genbank_file_basename)

shutil.copy2(genbank_file_src, genbank_file_dst)


In [32]:
from aisynbiopipeline.workflows.reference_utils import list_reference_genomes

list_reference_genomes()

['ADP1_Neidle_CDM.gbk',
 'ACN2821_CDM.gbk',
 'ACN2586_NSS.gbk',
 'ACN3500_IRZ.gbk']

In [ ]:
## To run breseq celery worksers, activate micromamba, then call worker script:
"""
export PATH="/opt/micromamba/bin/:$PATH"
eval "$(micromamba shell hook --shell bash)"
micromamba activate
micromamba activate aisynbio_env
python -m aisynbiopipeline.tasks.breseq_task 1
"""

In [ ]:
# Three breseq workers are running:

# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 1
# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 2

In [34]:
# Create breseq dir

short.create_subfolder('breseq')

In [38]:
short_manifest = short.create_manifest('received')
short_manifest

,sample_name,R1,R2
0,ADP1col1_ACN3500col1_50-50,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
1,ADP1col1_ACN3500col1_75-25,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
2,ADP1col1_ACN3500col1_90-10,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
3,ANL.stock.ACN2821.colony1,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
4,ANL.stock.ACN2821.colony2,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
5,ANL.stock.ACN2821.colony3,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
6,ANL.stock.ACN3500.colony1,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
7,ANL.stock.ACN3500.colony2,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
8,ANL.stock.ACN3500.colony3,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...


In [50]:
# Create seqsample groups

seqsamples2821 = [SeqSample(short, row['sample_name']) for _, row in short_manifest.iloc[3:6].iterrows()]
seqsamples3500 = [SeqSample(short, row['sample_name']) for _, row in short_manifest.iloc[6:].iterrows()]
seqsamplesMix = [SeqSample(short, row['sample_name']) for _, row in short_manifest.iloc[:3].iterrows()]
seqsamples3500_and_mix = [SeqSample(short, row['sample_name']) for _, row in short_manifest.iloc[[0,1,2,6,7,8]].iterrows()]

In [54]:
# Where should this code go?

# Specifies and assigns breseq parameters

def define_breseq_params(seqsample, ref_filename, poly=True, fold_coverage=300, num_processors=4):

    # Normalize Path → str - Celery tasks only accept certain input data types.
    def norm(x): return str(x) if isinstance(x, Path) else x
    
    breseq_params = {
        'read_paths': [norm(x) for x in seqsample.trimmed],
        'breseq_folder': norm(seqsample.breseq),
        'reference': ref_filename,
        'polymorphism_prediction': poly,
        'limit_fold_coverage': fold_coverage,
        'num_processors': num_processors
    }
    return breseq_params

In [55]:
# Submit tasks

results = []

for sample in seqsamples:
    result = client.send_task(
        'breseq.run',
        kwargs=define_breseq_params(sample, ref_filename='ADP1_Neidle_CDM.gbk'),
        queue='breseq'
    )
    results.append(result)

for sample in seqsamples2821:
    result = client.send_task(
        'breseq.run',
        kwargs=define_breseq_params(sample, ref_filename='ACN2821_CDM.gbk'),
        queue='breseq'
    )
    results.append(result)

for sample in seqsamples3500_and_mix:
    result = client.send_task(
        'breseq.run',
        kwargs=define_breseq_params(sample, ref_filename='ACN3500_IRZ.gbk'),
        queue='breseq'
    )
    results.append(result)
    

In [56]:
sum([(r.status=='SUCCESS') for r in results])

0

In [87]:
for i in results:
    print(i.status)
for i in results:
    print(i.r['output']) ## Check version_names for three different breseq runs

SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
SUCCESS
/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-01-13_S2SVV5/Plasmidsaurus_2026-01-13_S2SVV5_Illumina/breseq/ADP1col1_ACN3500col1_50-50/breseq_06bcbb18f5
/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-01-13_S2SVV5/Plasmidsaurus_2026-01-13_S2SVV5_Illumina/breseq/ADP1col1_ACN3500col1_75-25/breseq_06bcbb18f5
/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-01-13_S2SVV5/Plasmidsaurus_2026-01-13_S2SVV5_Illumina/breseq/ADP1col1_ACN3500col1_90-10/breseq_06bcbb18f5
/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-01-13_S2SVV5/Plasmidsaurus_2026-01-13_S2SVV5_Illumina/breseq/ANL.stock.ACN2821.colony1/breseq_06bcbb18f5
/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-01-

## 5. Breseq analysis

- Create breseq objects for each seqsample and aggregate summary counts.
- Generate mutation table for all samples and write to csv for import into Google Sheets.

In [76]:
from aisynbiopipeline.workflows.reference_utils import get_ref_genomes_path, genomic_region_from_features

genome2821 = os.path.join(get_ref_genomes_path(), 'ACN2821_CDM.gbk')
genome3500 = os.path.join(get_ref_genomes_path(), 'ACN3500_IRZ.gbk')

def get_region_parameter(genbank_file, feature_first, feature_last):
    genome, start, stop = genomic_region_from_features(genbank_file, feature_first, feature_last)
    region = genome + ":" + str(start) + "-" + str(stop)
    return region


In [108]:
from aisynbiopipeline.workflows.breseq import Breseq

def create_breseq_summary(seqsample_batch, version_name, output_path, regions=None):
    
    breseq_objects = []
    
    for s in seqsample_batch:
        breseq_folder = s.library.path / 'breseq' / s.sample_name/ version_name
        b = Breseq.from_existing(breseq_folder)
        breseq_objects.append(b)
    
    rows = []
    
    for b in breseq_objects:
        
        row = {}
        
        try:
            b.count_reads()
            b.count_mutations()
            b.avg_coverage
            if regions:
                for key, value in regions.items():
                    b.get_region_average_coverage(value)
        except Exception as e:
            row.update({'seqsample': getattr(b, 'title', None)})
            # row.update(parse_seqsample_name(getattr(b, 'title', None)))
            row.update({'error': str(e),
                        'input_read_count': None,
                        'used_read_count': None,
                        'mapped_read_count': None,
                        'consensus_mutation_count': None,
                        'polymorphism_mutation_count': None,
                        'average_cov': None,}
                        )
            if regions:
                for key, value in regions.items():
                    row.update({key: None})
            rows.append(row)
            print(f"Error loading Breseq from {b.title}: {e}")
            continue
    
        row['seqsample'] = getattr(b, 'title', None)
        # row.update(parse_seqsample_name(getattr(b, 'title', None)))
        row['error'] = None
        row['input_read_count'] = getattr(b, 'input_read_count', None)
        row['used_read_count'] = getattr(b, 'used_read_count', None)
        row['mapped_read_count'] = getattr(b, 'mapped_read_count', None)
        row['consensus_mutation_count'] = getattr(b, 'consensus_mutation_count', None)
        row['polymorphism_mutation_count'] = getattr(b, 'polymorphism_mutation_count', None)
        row['average_cov'] = getattr(b, 'avg_coverage', None)
        if regions:
            for key, value in regions.items():
                row[key] = getattr(b, 'get_region_average_coverage', None)(value)
        rows.append(row)
    
    breseq_summary = pd.DataFrame(rows)
    
    if regions:
        for key, value in regions.items():
            breseq_summary[key+'_CN'] = breseq_summary[key]/breseq_summary['average_cov']

    # Write breseq run summary to csv
    breseq_summary.to_csv(output_path)
    
    return breseq_summary

In [110]:
def create_html_comparison(seqsample_batch, version_name, output_path):

    from aisynbiopipeline.workflows.breseq import compare_gdiff

    breseq_objects = []
    
    for s in seqsample_batch:
        breseq_folder = s.library.path / 'breseq' / s.sample_name/ version_name
        b = Breseq.from_existing(breseq_folder)
        breseq_objects.append(b)

    reference = b.params.reference
    gdiffs = [b.gd_file for b in breseq_objects]

    table_format = 'html'
    html = compare_gdiff(reference, output_path, gdiffs, format=table_format)

    return html

In [80]:
regions = {
    'dgoA-Star': get_region_parameter(genome2821, 'dgoA-optimized-ADP1', 'dgoA-optimized-ADP1'),
    'ver_cassette': get_region_parameter(genome3500, 'omega KmR cassette', 'verR')
}

regions

/home/nspahr/.local/share/mamba/envs/aisynbio_env/lib/python3.12/site-packages/Bio/GenBank/Scanner.py:1537: BiopythonParserWarning: Attempting to parse malformed locus line:
'LOCUS       NC_005966     3599271 bp    DNA     circular UNA 29-JUL-2025\n'
Found locus 'NC_005966' size '3599271' residue_type 'DNA'
Some fields may be wrong.
  warnings.warn(


{'dgoA-Star': 'NC_005966:1677416-1678033',
 'ver_cassette': 'ACN3500_IRZ:941312-950627'}

### 2821 samples
- For ADP1 version, create summary csv and html comparison.
- For ACN2821 version, create summary csv (including dgoA coverage) and html comparison.
- Write 4 files to 2821 breseq analysis dir.

In [113]:
regions_to_include = ['dgoA-Star']
regions_sub = {key:regions[key] for key in regions_to_include if key in regions}

create_breseq_summary(seqsamples2821, 'breseq_06bcbb18f5', os.path.join(breseq_ACN2821_dir, 'mutation_summary_ADP1.csv'))
create_breseq_summary(seqsamples2821, 'breseq_066f7337ad', os.path.join(breseq_ACN2821_dir, 'mutation_summary_ACN2821.csv'), regions_sub)
create_html_comparison(seqsamples2821, 'breseq_06bcbb18f5', os.path.join(breseq_ACN2821_dir, 'mutation_comparison_ADP1.html'))
create_html_comparison(seqsamples2821, 'breseq_066f7337ad', os.path.join(breseq_ACN2821_dir, 'mutation_comparison_ACN2821.html'))

breseq 0.39.0     http://barricklab.org/breseq

Active Developers: Barrick JE, Deatherage DE
Contact:           <jeffrey.e.barrick@gmail.com>

breseq is free software; you can redistribute it and/or modify it under the
terms the GNU General Public License as published by the Free Software 
Foundation; either version 2, or (at your option) any later version.

Copyright (c) 2008-2010 Michigan State University
Copyright (c) 2011-2022 The University of Texas at Austin

If you use breseq in your research, please cite:

  Deatherage, D.E., Barrick, J.E. (2014) Identification of mutations
  in laboratory-evolved microbes from next-generation sequencing
  data using breseq. Methods Mol. Biol. 1151: 165–188.

If you use structural variation (junction) predictions, please cite:

  Barrick, J.E., Colburn, G., Deatherage D.E., Traverse, C.C.,
  Strand, M.D., Borges, J.J., Knoester, D.B., Reba, A., Meyer, A.G. 
  (2014) Identifying structural variation in haploid microbial genomes 
  from short-rea

'/storage/nspahr/lib_analysis/Plasmidsaurus_2026-01-13_S2SVV5/breseq_analysis_ACN2821/mutation_comparison_ACN2821.html'

In [134]:
# Symlinks to ACN2821 output
output_symlinks_dir = os.path.join(breseq_ACN2821_dir, 'symlinks_to_breseq_ACN2821_output')
os.makedirs(output_symlinks_dir)

for seqsample in seqsamples2821:
    path_to_folder = seqsample.breseq / 'breseq_066f7337ad'
    dst = os.path.join(output_symlinks_dir, seqsample.sample_name)
    os.symlink(path_to_folder, dst)

### 3500 samples
- For ADP1 version, create summary csv and html comparison.
- For ACN3500 version, create summary csv (including ver cassette coverage) and html comparison.
- Write 4 files to 3500 breseq analysis dir.
- Create subfolder with symlinks to breseq output.


In [116]:
regions_to_include = ['ver_cassette']
regions_sub = {key:regions[key] for key in regions_to_include if key in regions}

create_breseq_summary(seqsamples3500, 'breseq_06bcbb18f5', os.path.join(breseq_ACN3500_dir, 'mutation_summary_ADP1.csv'))
create_breseq_summary(seqsamples3500, 'breseq_0d5e3dff7b', os.path.join(breseq_ACN3500_dir, 'mutation_summary_ACN3500.csv'), regions_sub)
create_html_comparison(seqsamples3500, 'breseq_06bcbb18f5', os.path.join(breseq_ACN3500_dir, 'mutation_comparison_ADP1.html'))
create_html_comparison(seqsamples3500, 'breseq_0d5e3dff7b', os.path.join(breseq_ACN3500_dir, 'mutation_comparison_ACN3500.html'))

breseq 0.39.0     http://barricklab.org/breseq

Active Developers: Barrick JE, Deatherage DE
Contact:           <jeffrey.e.barrick@gmail.com>

breseq is free software; you can redistribute it and/or modify it under the
terms the GNU General Public License as published by the Free Software 
Foundation; either version 2, or (at your option) any later version.

Copyright (c) 2008-2010 Michigan State University
Copyright (c) 2011-2022 The University of Texas at Austin

If you use breseq in your research, please cite:

  Deatherage, D.E., Barrick, J.E. (2014) Identification of mutations
  in laboratory-evolved microbes from next-generation sequencing
  data using breseq. Methods Mol. Biol. 1151: 165–188.

If you use structural variation (junction) predictions, please cite:

  Barrick, J.E., Colburn, G., Deatherage D.E., Traverse, C.C.,
  Strand, M.D., Borges, J.J., Knoester, D.B., Reba, A., Meyer, A.G. 
  (2014) Identifying structural variation in haploid microbial genomes 
  from short-rea

'/storage/nspahr/lib_analysis/Plasmidsaurus_2026-01-13_S2SVV5/breseq_analysis_ACN3500/mutation_comparison_ACN3500.html'

In [ ]:
# Symlinks to ACN3500 output
output_symlinks_dir = os.path.join(breseq_ACN3500_dir, 'symlinks_to_breseq_ACN3500_output')
os.makedirs(output_symlinks_dir)

for seqsample in seqsamples3500:
    path_to_folder = seqsample.breseq / 'breseq_0d5e3dff7b'
    dst = os.path.join(output_symlinks_dir, seqsample.sample_name)
    os.symlink(path_to_folder, dst)

In [124]:
# Symlinks to ADP1 output
output_symlinks_dir = os.path.join(breseq_ACN3500_dir, 'symlinks_to_breseq_ADP1_output')
os.makedirs(output_symlinks_dir)

for seqsample in seqsamples3500:
    path_to_folder = seqsample.breseq / 'breseq_06bcbb18f5'
    dst = os.path.join(output_symlinks_dir, seqsample.sample_name)
    os.symlink(path_to_folder, dst)

### Mixture samples
- For ADP1 version, create summary csv and html comparison.
- For ACN3500 version, create summary csv (including ver cassette coverage) and html comparison.
- Write 4 files to mixtures breseq analysis dir.

In [117]:
regions_to_include = ['ver_cassette']
regions_sub = {key:regions[key] for key in regions_to_include if key in regions}

create_breseq_summary(seqsamplesMix, 'breseq_06bcbb18f5', os.path.join(breseq_mixtures_dir, 'mutation_summary_ADP1.csv'))
create_breseq_summary(seqsamplesMix, 'breseq_0d5e3dff7b', os.path.join(breseq_mixtures_dir, 'mutation_summary_ACN3500.csv'), regions_sub)
create_html_comparison(seqsamplesMix, 'breseq_06bcbb18f5', os.path.join(breseq_mixtures_dir, 'mutation_comparison_ADP1.html'))
create_html_comparison(seqsamplesMix, 'breseq_0d5e3dff7b', os.path.join(breseq_mixtures_dir, 'mutation_comparison_ACN3500.html'))

breseq 0.39.0     http://barricklab.org/breseq

Active Developers: Barrick JE, Deatherage DE
Contact:           <jeffrey.e.barrick@gmail.com>

breseq is free software; you can redistribute it and/or modify it under the
terms the GNU General Public License as published by the Free Software 
Foundation; either version 2, or (at your option) any later version.

Copyright (c) 2008-2010 Michigan State University
Copyright (c) 2011-2022 The University of Texas at Austin

If you use breseq in your research, please cite:

  Deatherage, D.E., Barrick, J.E. (2014) Identification of mutations
  in laboratory-evolved microbes from next-generation sequencing
  data using breseq. Methods Mol. Biol. 1151: 165–188.

If you use structural variation (junction) predictions, please cite:

  Barrick, J.E., Colburn, G., Deatherage D.E., Traverse, C.C.,
  Strand, M.D., Borges, J.J., Knoester, D.B., Reba, A., Meyer, A.G. 
  (2014) Identifying structural variation in haploid microbial genomes 
  from short-rea

'/storage/nspahr/lib_analysis/Plasmidsaurus_2026-01-13_S2SVV5/breseq_analysis_mixtures/mutation_comparison_ACN3500.html'

## 6. Rerunning breseq with new reference genome version 
### Short reads: Breseq (population mode)

Copy 3500 reference genome to reference genome folder.

Run breseq:
ACN3500 samples against ACN3500_NSS.gbk ref genome

In [137]:
# Copy reference genome into ref genome folder.

from aisynbiopipeline.workflows.reference_utils import get_ref_genomes_path, list_reference_genomes

genbank_file_src = '/storage/nspahr/tmp/ACN3500_NSS.gbk'
genbank_file_basename = os.path.basename(genbank_file_src)
genbank_file_dst = os.path.join(get_ref_genomes_path(), genbank_file_basename)

shutil.copy2(genbank_file_src, genbank_file_dst)


'/storage/synbio/ai_synbio_data/reference_data/genomes/ACN3500_NSS.gbk'

In [138]:
from aisynbiopipeline.workflows.reference_utils import list_reference_genomes

list_reference_genomes()

['ADP1_Neidle_CDM.gbk',
 'ACN2821_CDM.gbk',
 'ACN2586_NSS.gbk',
 'ACN3500_IRZ.gbk',
 'ACN3500_NSS.gbk']

In [ ]:
## To run breseq celery worksers, activate micromamba, then call worker script:
"""
cd ~/code/AISynbioPipeline
export PATH="/opt/micromamba/bin/:$PATH"
eval "$(micromamba shell hook --shell bash)"
micromamba activate
micromamba activate aisynbio_env
python -m aisynbiopipeline.tasks.breseq_task 1
"""

In [ ]:
# Three breseq workers are running:

# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 1
# (aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ python -m aisynbiopipeline.tasks.breseq_task 2

In [139]:
# Where should this code go?

# Specifies and assigns breseq parameters

def define_breseq_params(seqsample, ref_filename, poly=True, fold_coverage=300, num_processors=4):

    # Normalize Path → str - Celery tasks only accept certain input data types.
    def norm(x): return str(x) if isinstance(x, Path) else x
    
    breseq_params = {
        'read_paths': [norm(x) for x in seqsample.trimmed],
        'breseq_folder': norm(seqsample.breseq),
        'reference': ref_filename,
        'polymorphism_prediction': poly,
        'limit_fold_coverage': fold_coverage,
        'num_processors': num_processors
    }
    return breseq_params

In [142]:
breseq_objects = []

for s in seqsamples3500:
    breseq_folder = s.library.path / 'breseq' / s.sample_name/ 'breseq_0d5e3dff7b'
    b = Breseq.from_existing(breseq_folder)
    breseq_objects.append(b)

In [144]:
for b in breseq_objects:
    print(b.reference)   

ACN3500_IRZ.gbk
ACN3500_IRZ.gbk
ACN3500_IRZ.gbk


In [145]:
# Submit tasks

results = []

for sample in seqsamples3500:
    result = client.send_task(
        'breseq.run',
        kwargs=define_breseq_params(sample, ref_filename='ACN3500_NSS.gbk'),
        queue='breseq'
    )
    results.append(result)
    

In [150]:
sum([(r.status=='SUCCESS') for r in results])

3

In [151]:
for i in results:
    print(i.status)
for i in results:
    print(i.result) ## Check version_name to make sure it's different from previous version name

SUCCESS
SUCCESS
SUCCESS
{'status': 'success', 'output': '/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-01-13_S2SVV5/Plasmidsaurus_2026-01-13_S2SVV5_Illumina/breseq/ANL.stock.ACN3500.colony1/breseq_df1972644b'}
{'status': 'success', 'output': '/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-01-13_S2SVV5/Plasmidsaurus_2026-01-13_S2SVV5_Illumina/breseq/ANL.stock.ACN3500.colony2/breseq_df1972644b'}
{'status': 'success', 'output': '/storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/Plasmidsaurus_2026-01-13_S2SVV5/Plasmidsaurus_2026-01-13_S2SVV5_Illumina/breseq/ANL.stock.ACN3500.colony3/breseq_df1972644b'}


In [152]:
breseq_objects = []

for s in seqsamples3500:
    breseq_folder = s.library.path / 'breseq' / s.sample_name/ 'breseq_df1972644b'
    b = Breseq.from_existing(breseq_folder)
    breseq_objects.append(b)

In [153]:
for b in breseq_objects:
    print(b.reference)   

ACN3500_NSS.gbk
ACN3500_NSS.gbk
ACN3500_NSS.gbk


### 3500 samples
- Create summary csv (including ver cassette coverage) and html comparison.
- Write files to 3500 breseq analysis subdir (breseq_analysis_ACN3500_NSS/).
- Create subfolder with symlinks to breseq output.

In [154]:
breseq_ACN3500_dir = home_dir + '/breseq_analysis_ACN3500/breseq_analysis_ACN3500_NSS/'
os.makedirs(breseq_ACN3500_dir)

In [159]:
genome3500_NSS = os.path.join(get_ref_genomes_path(), 'ACN3500_NSS.gbk')

In [160]:
regions = {
    'ver_cassette': get_region_parameter(genome3500_NSS, 'omega KmR cassette', 'verR')
}

regions

{'ver_cassette': 'ACN3500_NSS:941311-950626'}

In [162]:
regions_to_include = ['ver_cassette']
regions_sub = {key:regions[key] for key in regions_to_include if key in regions}

create_breseq_summary(seqsamples3500, 'breseq_df1972644b', os.path.join(breseq_ACN3500_dir, 'mutation_summary_ACN3500_NSS.csv'), regions_sub)
create_html_comparison(seqsamples3500, 'breseq_df1972644b', os.path.join(breseq_ACN3500_dir, 'mutation_comparison_ACN3500_NSS.html'))

breseq 0.39.0     http://barricklab.org/breseq

Active Developers: Barrick JE, Deatherage DE
Contact:           <jeffrey.e.barrick@gmail.com>

breseq is free software; you can redistribute it and/or modify it under the
terms the GNU General Public License as published by the Free Software 
Foundation; either version 2, or (at your option) any later version.

Copyright (c) 2008-2010 Michigan State University
Copyright (c) 2011-2022 The University of Texas at Austin

If you use breseq in your research, please cite:

  Deatherage, D.E., Barrick, J.E. (2014) Identification of mutations
  in laboratory-evolved microbes from next-generation sequencing
  data using breseq. Methods Mol. Biol. 1151: 165–188.

If you use structural variation (junction) predictions, please cite:

  Barrick, J.E., Colburn, G., Deatherage D.E., Traverse, C.C.,
  Strand, M.D., Borges, J.J., Knoester, D.B., Reba, A., Meyer, A.G. 
  (2014) Identifying structural variation in haploid microbial genomes 
  from short-rea

'/storage/nspahr/lib_analysis/Plasmidsaurus_2026-01-13_S2SVV5/breseq_analysis_ACN3500/breseq_analysis_ACN3500_NSS/mutation_comparison_ACN3500_NSS.html'

In [163]:
# Symlinks to ACN3500 output
output_symlinks_dir = os.path.join(breseq_ACN3500_dir, 'symlinks_to_breseq_ACN3500_NSS_output')
os.makedirs(output_symlinks_dir)

for seqsample in seqsamples3500:
    path_to_folder = seqsample.breseq / 'breseq_df1972644b'
    dst = os.path.join(output_symlinks_dir, seqsample.sample_name)
    os.symlink(path_to_folder, dst)